In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("../data/raw/sri_lanka_survey.csv")
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Loaded: 20 rows × 29 columns
Columns: ['Timestamp', '1. What is your gender?', '2. What is your age range?', '3. Which district are you from?', '4. What is your occupation?', '5. What is your approximate monthly household spending on products? (LKR)', '6. How often do you shop for products?', '06. Does your cultural background influence what products you buy?', 'Q07.  Clothing & Apparel\nWhat do you mainly check before buying a clothing & apparel product?', 'Q08. Clothing & Apparel  \nWhat is your main reason for buying clothing & apparel products?', 'Q09. Clothing & Apparel   \nHow much does emotional appeal (how it makes you feel) influence your purchase of clothing & apparel products?', 'Q10.   Beauty & Personal Care\nWhat do you mainly check before buying a beauty & personal care product?', 'Q11.   Beauty & Personal Care\n What is your main reason for buying beauty & personal care products?', 'Q12.   Beauty & Personal Care\nHow much does emotional appeal (how it makes you feel) inf

In [4]:
COL_GENDER    = '1. What is your gender?'
COL_AGE       = '2. What is your age range?'
COL_DISTRICT  = '3. Which district are you from?'
COL_OCC       = '4. What is your occupation?'
COL_SPENDING  = '5. What is your approximate monthly household spending on products? (LKR)'
COL_CULTURE   = '06. Does your cultural background influence what products you buy?'

In [5]:
SCALE_COLS = [
    'Q09. Clothing & Apparel   \nHow much does emotional appeal (how it makes you feel) influence your purchase of clothing & apparel products?',
    'Q12.   Beauty & Personal Care\nHow much does emotional appeal (how it makes you feel) influence your purchase of beauty & personal care products?',
    'Q15.  Electronics\nHow much does emotional appeal (how it makes you feel) influence your purchase of electronics products?',
    'Q18.    Grocery & Food  \n  How much does emotional appeal (how it makes you feel) influence your purchase of grocery & food products?',
    'Q21.  Baby Products\nHow much does emotional appeal (how it makes you feel) influence your purchase of baby products products?',
    'Q24.  Pet Products\nHow much does emotional appeal (how it makes you feel) influence your purchase of pet products products?',
    'Q27.  Sports & Fitness\nHow much does emotional appeal (how it makes you feel) influence your purchase of sports & fitness products?',
]

In [6]:
REASON_COLS = [
    'Q08. Clothing & Apparel  \nWhat is your main reason for buying clothing & apparel products?',
    'Q11.   Beauty & Personal Care\n What is your main reason for buying beauty & personal care products?',
    'Q14.  Electronics \nWhat is your main reason for buying electronics products?',
    'Q17.    Grocery & Food\nWhat is your main reason for buying grocery & food products?',
    'Q20. Baby Products\nWhat is your main reason for buying baby products products?',
    'Q23.  Pet Products\nWhat is your main reason for buying pet products products?',
    'Q26.  Sports & Fitness\nWhat is your main reason for buying sports & fitness products?',
]

In [7]:
CHECK_COLS = [
    'Q07.  Clothing & Apparel\nWhat do you mainly check before buying a clothing & apparel product?',
    'Q10.   Beauty & Personal Care\nWhat do you mainly check before buying a beauty & personal care product?',
    'Q13.  Electronics\nWhat do you mainly check before buying a electronics product?',
    'Q16.   Grocery & Food\nWhat do you mainly check before buying a grocery & food product?',
    'Q19.  Baby Products \n  What do you mainly check before buying a baby products product?',
    'Q22.  Pet Products\nWhat do you mainly check before buying a pet products product?',
    'Q25.  Sports & Fitness\nWhat do you mainly check before buying a sports & fitness product?',
]

In [8]:
all_expected = [COL_GENDER, COL_AGE, COL_DISTRICT, COL_OCC,
                COL_SPENDING, COL_CULTURE] + SCALE_COLS + REASON_COLS + CHECK_COLS
 
missing = [c for c in all_expected if c not in df.columns]
if missing:
    print(f"WARNING: {len(missing)} columns not found: {missing[:3]}...")
else:
    print("All expected columns found ")

All expected columns found 


In [ ]:
#encoders for demographic features
def enc_gender(v):
    v = str(v).strip().lower()
    return 1.0 if 'female' in v else 0.0

In [10]:
def enc_age(v):
    m = {'under 18':0,'18 – 24':1,'25 – 34':2,
         '35 – 44':3,'45 – 54':4,'55 and above':5}
    return float(m.get(str(v).strip(), 2))

In [11]:
def enc_district(d):
    urban = ['colombo','gampaha','kandy','galle','kalutara',
             'kurunegala','ratnapura','matara','jaffna']
    return 1.0 if str(d).lower().strip() in urban else 0.0

In [13]:
def enc_occupation(v):
    m = {
        'student': 0.7,
        'homemaker': 0.5,
        'private sector employee': 0.4,
        'self-employed/business owner': 0.3,
        'other': 0.5,
    }
    return m.get(str(v).strip().lower(), 0.5)

In [14]:
def enc_spending(v):
    m = {
        'less than rs. 5,000':    0.0,
        'rs. 5,000 – rs. 15,000': 0.25,
        'rs. 15,001 – rs. 30,000':0.5,
        'rs. 30,001 – rs. 50,000':0.75,
        'more than rs. 50,000':   1.0,
    }
    return m.get(str(v).strip().lower(), 0.5)

In [15]:
def enc_culture(v):
    v = str(v).strip().lower()
    if 'strongly' in v: return 1.0
    if 'no' in v:       return 0.0
    return 0.5 

In [16]:
#apply for all 
df['gender_enc']      = df[COL_GENDER].apply(enc_gender)
df['age_enc']         = df[COL_AGE].apply(enc_age)
df['environment_enc'] = df[COL_DISTRICT].apply(enc_district)
df['occupation_enc']  = df[COL_OCC].apply(enc_occupation)
df['spending_enc']    = df[COL_SPENDING].apply(enc_spending)
df['culture_enc']     = df[COL_CULTURE].apply(enc_culture)
 
print("Demographic features encoded ")

Demographic features encoded 


In [ ]:
scale_matrix = pd.DataFrame(index=df.index)
for col in SCALE_COLS:
    scale_matrix[col] = pd.to_numeric(df[col], errors='coerce').fillna(3)